<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/Vize%20Randevu%20kontrol%20edici.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Playwright için gerekli sistem bağımlılıklarını kurun
!apt-get update && apt-get install -y libatk-bridge2.0-0 libgtk-3-0 libgbm-dev libxkbcommon-dev

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,849 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libatk-bridge2.0-0 is already the newest version (

In [9]:
# Playwright Python kütüphanesini ve Chromium'u kurun
!pip -q install playwright
!playwright install chromium

### E-posta Bildirim Ayarları

E-posta gönderebilmek için Gmail gibi bir sağlayıcıdan "Uygulama Şifresi" (App Password) oluşturmanız ve bunu Colab'in "Gizli Anahtarlar" (Secrets) bölümüne eklemeniz gerekmektedir.

Sol paneldeki 🔑 simgesine tıklayın ve şu anahtarları ekleyin:

1.  `SENDER_EMAIL`: E-postaları gönderecek olan e-posta adresiniz (örn: `you@gmail.com`).
2.  `SENDER_PASSWORD`: Gmail için oluşturduğunuz Uygulama Şifreniz (normal şifreniz değil). Bu, 16 karakterli bir dizedir.
3.  `RECEIVER_EMAIL`: Bildirimi alacak e-posta adresi (örn: `your_other_email@example.com`).

Bu bilgileri ekledikten sonra aşağıdaki kodu çalıştırarak e-posta gönderme işlevini ayarlayabilirsiniz.

In [10]:
import smtplib
from email.mime.text import MIMEText
from google.colab import userdata

# Colab Secrets'tan e-posta bilgilerini çekin
SENDER_EMAIL = userdata.get('SENDER_EMAIL')
SENDER_PASSWORD = userdata.get('SENDER_PASSWORD')
RECEIVER_EMAIL = userdata.get('RECEIVER_EMAIL')

def send_email_notification(subject, body):
    if not SENDER_EMAIL or not SENDER_PASSWORD or not RECEIVER_EMAIL:
        print("E-posta kimlik bilgileri yapılandırılmamış. E-posta bildirimi atlanıyor.")
        return

    msg = MIMEText(body, 'html', 'utf-8')
    msg['Subject'] = subject
    msg['From'] = SENDER_EMAIL
    msg['To'] = RECEIVER_EMAIL

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(SENDER_EMAIL, SENDER_PASSWORD)
            smtp.send_message(msg)
        print("E-posta bildirimi başarıyla gönderildi!")
    except Exception as e:
        print(f"E-posta bildirimi gönderilemedi: {e}")

In [12]:
import asyncio, re
from playwright.async_api import async_playwright

URL = "https://appointment.as-visa.com/tr/istanbul-bireysel-basvuru"
CHECK_EVERY_SECONDS = 300

NO_QUOTA_TEXT = "Aktif Randevu Kotamız bulunmamaktadır"
BOT_KEYWORDS = [
    "cloudflare", "just a moment", "checking your browser", "attention required",
    "captcha", "recaptcha", "hcaptcha", "turnstile", "cf-chl"
]

def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

async def check_once_get_status():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        page = await browser.new_page(
            locale="tr-TR",
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
            viewport={"width": 1280, "height": 720},
        )

        await page.goto(URL, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(6000)

        final_url = page.url
        title = await page.title()

        body_text = await page.locator("body").inner_text(timeout=30000)
        body_text_n = normalize_text(body_text)

        current_no_quota = NO_QUOTA_TEXT.lower() in body_text_n.lower()
        bot_hits = [k for k in BOT_KEYWORDS if k in body_text_n.lower()]

        print("\n" + "="*80)
        print("Final URL:", final_url)
        print("Title   :", title)
        print("NO_QUOTA visible?:", current_no_quota)
        print("Bot keyword hits :", bot_hits if bot_hits else "None")
        print("-"*80)
        print("BODY TEXT (first 1000 chars):")
        print(body_text_n[:1000])
        print("="*80)

        await browser.close()
        return current_no_quota, body_text_n

async def main_loop_with_email():
    # Başlangıçta sitenin mevcut durumunu kontrol et ve e-posta gönder
    initial_no_quota, initial_body_text = await check_once_get_status()

    subject_initial = "VİZE RANDEVU İZLEYİCİ BAŞLADI - İlk Durum"
    email_body_initial = f"""
    <html>
        <body>
            <p>Vize randevu izleyici başarıyla başlatıldı!</p>
            <p>Şu anki durum: **{'Aktif Randevu Kotası Yok' if initial_no_quota else 'Randevu Kotası Açık Olabilir'}**</p>
            <p>Web sitesi içeriği (ilk 1000 karakter):</p>
            <pre>{initial_body_text[:1000]}</pre>
            <p><a href=\"{URL}\">{URL}</a></p>
        </body>
    </html>
    """
    send_email_notification(subject_initial, email_body_initial)
    print("Başlangıç durumu e-posta bildirimi gönderildi.")

    previous_no_quota_status = initial_no_quota

    while True:
        try:
            current_no_quota, _ = await check_once_get_status()

            # Eğer daha önce kota yokken şimdi kota varsa e-posta gönder
            if previous_no_quota_status == True and current_no_quota == False:
                subject = "VİZE RANDEVU KOTASI AÇILDI!"
                email_body = f"""
                <html>
                    <body>
                        <p>İyi haber! <a href=\"{URL}\">{URL}</a> adresinde vize randevu kotası açılmış olabilir!</p>
                        <p>'Aktif Randevu Kotamız bulunmamaktadır' metni artık sayfada görünmüyor.</p>
                        <p>Lütfen web sitesini hemen kontrol edin.</p>
                    </body>
                </html>
                """
                send_email_notification(subject, email_body)
                print("Randevu kotası açıldığı için e-posta bildirimi gönderildi.")
            elif previous_no_quota_status == False and current_no_quota == True:
                print("Kota vardı ama şimdi kapandı.")
            elif previous_no_quota_status == True and current_no_quota == True:
                print("Hala randevu kotası bulunmamaktadır.")
            else: # previous_no_quota_status == False and current_no_quota == False
                print("Randevu kotası henüz açılmadı.")

            # Mevcut durumu bir sonraki döngü için kaydet
            previous_no_quota_status = current_no_quota

        except Exception as e:
            print("Hata:", repr(e))
        await asyncio.sleep(CHECK_EVERY_SECONDS)

# E-posta gönderme işlevinin global olarak tanımlandığını varsayıyoruz.
# Bu hücreyi çalıştırmadan önce yukarıdaki e-posta kurulum hücrelerini çalıştırdığınızdan emin olun.

await main_loop_with_email()


Final URL: https://appointment.as-visa.com/tr/istanbul-bireysel-basvuru
Title   : AS VISA SOLUTIONS | Macaristan, Portekiz, Norveç ve Slovenya Schengen Vize Başvuruları için Tek Yetkili Kurum
NO_QUOTA visible?: False
Bot keyword hits : None
--------------------------------------------------------------------------------
BODY TEXT (first 1000 chars):
Randevu Al Başvuru & Randevu Takibi Randevu Kayıt Sorgula İstanbul Piyalepaşa Randevu Talebi Oluşturma İstanbul Piyalepaşa Randevu Talebi Oluşturma Aktif Randevu Kotası Bulunmamaktadır Sayın Başvuru Sahibimiz, seçmiş olduğunuz işlem merkezi ve vize türü için an itibarıyla uygun randevu kotası bulunmamaktadır. Kontenjanlarımız periyodik olarak güncellenmekte olup, randevu modülümüzü ilerleyen günlerde tekrar ziyaret etmenizi rica ederiz. Anlayışınız için teşekkür eder, iyi günler dileriz.
E-posta bildirimi başarıyla gönderildi!
Başlangıç durumu e-posta bildirimi gönderildi.

Final URL: https://appointment.as-visa.com/tr/istanbul-bireysel-ba

CancelledError: 